# AELIONIX BLACKFORGE — Phase 10 Colab Validation

This notebook performs a deterministic, one-click validation of the **Identity &
Directory Security Capability Foundation**.

It exercises the full `blackforge.identity` pipeline on the mock **AELIONIX-CORP**
directory:

* **directory_discovery / identity_inventory / group_inventory** — the directory
  service, its identities (human/service/computer), and security groups
* **role_inventory / permission_inventory / resource_inventory** — authorization
  containers, permission rights, and the resources they apply to
* **membership_observation / role_assignment_observation** — which identities are
  in which groups and hold which roles (duplicates deterministically collapsed)
* **permission_assignment_observation** — permissions granted through an
  identity's roles
* **relationship_analysis** — structural edges only (`member_of` / `has_role` /
  `has_permission` / `applies_to`); no attack-graph vocabulary
* **metadata_observation** — directional attributes with authoritative-vs-
  inferred sources and explicit contradiction surfacing (never silent overwrite)

Every capability runs the same guarded pipeline: request validation, scope /
authorization, identity resolution, **credential-redacted mock transport**
(no real directory is ever queried or mutated), normalization, evidence
persistence, world-model materialization, and best-effort memory linking.
Credential-like fields (password hashes, session tokens, API keys) are stripped
before any artifact or observation row is persisted.

> Run all cells top-to-bottom. No GPU, no external services, no credentials.
> The notebook fails loudly on any check.

---

In [ ]:
import sys
import platform

print("Blackforge Phase 10 Colab Validation (Identity & Directory Security Capability Foundation)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")

---

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import shutil

# -- Configuration (edit here if fork changes) ---------------------------
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
REPO_DIR = Path("/content/blackforge")
# -----------------------------------------------------------------------

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

---

In [ ]:
import subprocess

try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("Commit:", commit)
except Exception as e:
    print("Commit unavailable (expected in scratch checkouts):", e)

---

In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]" --quiet

---

In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.capabilities.registry",
    "blackforge.authorization",
    "blackforge.scope.models",
    "blackforge.evidence",
    "blackforge.evidence.models",
    "blackforge.evidence.store",
    "blackforge.evidence.repository",
    "blackforge.world_model",
    "blackforge.world_model.models",
    "blackforge.world_model.query",
    "blackforge.world_model.repository",
    "blackforge.world_model.store",
    "blackforge.identity",
    "blackforge.identity.models",
    "blackforge.identity.transport",
    "blackforge.identity.redaction",
    "blackforge.identity.evidence",
    "blackforge.identity.normalization",
    "blackforge.identity.capabilities",
    "blackforge.identity.materializer",
    "blackforge.identity.engine",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} — {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("Identity module imports: PASS")

---

In [ ]:
import subprocess
import sys

print("Running automated test suite...")
# The LLM/torch-heavy files are excluded: importing the HF provider pulls
# ~2GB of torch memory and can SIGKILL the kernel on CPU runtimes. Those
# tests are validated locally and in the Phase 1 notebook.
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")

---

In [ ]:
import os
from pathlib import Path

DBROOT = Path("data/phase10_colab").resolve()
DBROOT.mkdir(parents=True, exist_ok=True)
os.environ["BLACKFORGE_DB_PATH"] = str(DBROOT / "blackforge.db")
os.environ["BLACKFORGE_MEMORY_DB_PATH"] = str(DBROOT / "memory.db")
os.environ["BLACKFORGE_EVIDENCE_DB_PATH"] = str(DBROOT / "evidence.db")
os.environ["BLACKFORGE_WORLD_MODEL_DB_PATH"] = str(DBROOT / "world_model.db")
for _p in (DBROOT / "evidence.db", DBROOT / "world_model.db"):
    _p.unlink(missing_ok=True)

from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
for key in ("config_loaded", "mission_manager_ready", "capability_registry_ready",
            "memory_ready", "evidence_store_ready", "evidence_memory_link_ready",
            "world_model_ready", "recon_ready", "webapi_ready", "auth_ready",
            "business_logic_ready", "network_ready", "identity_ready",
            "authorization_ready", "model_router_ready"):
    assert verification[key], f"{key} must be True"
assert verification["identity_ready"] is True, "identity_ready must be True (11 typed capabilities)"
assert len(app.capability_registry.list_capabilities()) == 61

BOOTSTRAP_OK = app.healthy() and bool(verification["identity_ready"])

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap (identity_ready, 61 registered capabilities): PASS")

---

In [ ]:
from blackforge.identity.capabilities import build_identity_capabilities
from blackforge.identity.models import IdentityMode, IdentityRequest
from blackforge.identity.transport import IDENTITY_DIRECTORY as DIR
from blackforge.identity.transport import IDENTITY_DIRECTORY_DNS as DNS
from blackforge.scope.models import TargetScope, Target, detect_target_type
from blackforge.core.types import RiskLevel, TargetType

def _target(value: str) -> Target:
    return Target(value=value, target_type=detect_target_type(value))

MID = "mission_phase10_id"
_ERROR_TARGETS = [
    "SNAIL-DIR", "BURSTY-DIR", "LOCKED-DIR", "GARBLED-DIR",
    "FABRICATED-DIR", "OTHER-CORP",
]
ALL = [
    DIR, DNS, "AELIONIX-CORP\\alice", "alice@aelionix-corp.local",
    "alice", "bob", "api-service", "ghost-identity",
] + _ERROR_TARGETS
scope = TargetScope(
    mission_id=MID,
    allowed_targets=[_target(t) for t in ALL],
    max_risk_level=RiskLevel.HIGH,
)
req = IdentityRequest(
    mission_id=MID, session_id="ses_phase10_id", scope=scope,
    mode=IdentityMode.CONTROLLED, max_observations=500, timeout_seconds=30.0,
)

engine = app.identity_engine
assert engine is not None and len(engine.capabilities) == 11
expected = sorted([
    "identity.directory_discovery",
    "identity.identity_inventory",
    "identity.group_inventory",
    "identity.role_inventory",
    "identity.permission_inventory",
    "identity.resource_inventory",
    "identity.membership_observation",
    "identity.role_assignment_observation",
    "identity.permission_assignment_observation",
    "identity.relationship_analysis",
    "identity.metadata_observation",
])
ids_seen = sorted(c.capability_id for c in engine.capabilities)
assert ids_seen == expected, (ids_seen, expected)
print("Registered identity capabilities:", ", ".join(c.capability_id for c in engine.capabilities))

_meta_by_id = {c.capability_id: c.meta() for c in engine.capabilities}
for capability_id in expected:
    meta = _meta_by_id[capability_id]
    risk = meta.risk_level.value
    mode = meta.mode.value
    assert risk == "low", capability_id
    assert mode == "passive", capability_id
    assert meta.world_model, f"{capability_id} must materialize into the world model"
    assert TargetType.ASSET in meta.supported_target_types, capability_id
    print(
        f"  {meta.id:<38} risk={risk:<7} mode={mode:<8} "
        f"targets={[t.value for t in meta.supported_target_types]}"
    )
CAPS_OK = True

---

In [ ]:
from blackforge.evidence.models import EvidenceRelation, EvidenceStatus, EvidenceType
from blackforge.core.types import Confidence
from blackforge.world_model.query import RelationshipQuery, WorldQuery
from blackforge.world_model.models import EntityType, WorldLifecycle

# --- deterministic pipeline over every capability -------------------------
r_dir  = engine.discover_directories(req, DIR)
r_inv  = engine.inventory_identities(req, DIR)
r_grp  = engine.inventory_groups(req, DIR)
r_rol  = engine.inventory_roles(req, DIR)
r_perm = engine.inventory_permissions(req, DIR)
r_res  = engine.inventory_resources(req, DIR)
r_mem  = engine.observe_membership(req, DIR, identity="alice")
r_role = engine.observe_role_assignment(req, DIR, identity="bob")
r_pa   = engine.observe_permission_assignment(req, DIR, identity="bob")
r_rel  = engine.analyze_relationships(req, DIR, identity="alice")
r_meta = engine.observe_metadata(req, DIR, identity="alice")

pipeline_runs = [r_dir, r_inv, r_grp, r_rol, r_perm, r_res, r_mem,
                 r_role, r_pa, r_rel, r_meta]
from blackforge.identity.models import IdentityStatus

for r in pipeline_runs:
    assert r.authorized is True, r.capability_id
    assert len(r.evidence_ids) >= 1, r.capability_id
statuses = []
for r in pipeline_runs:
    statuses.append(f"{r.capability_id.split('.')[-1]}={r.status.value}({len(r.observations)})")
print("All 11 identity capabilities executed")
print("Statuses:", " ".join(statuses))

assert r_dir.observation_count == 1
assert r_inv.observation_count == 5   # alice, bob, build-service, api-service, web-server-01$
assert r_grp.observation_count == 4   # engineering, operations, administrators, read-only
assert r_rol.observation_count == 4   # application-admin, deployment-operator, viewer, service-operator
assert r_perm.observation_count == 4  # deploy, manage, read, view_logs
assert r_res.observation_count == 4   # production-api, internal-dashboard, deployment-system, database-cluster
assert r_mem.observation_count == 1   # duplicate membership row deterministically collapsed
assert r_mem.status == IdentityStatus.PARTIAL
assert any("collapsed duplicate membership" in w for w in r_mem.warnings), r_mem.warnings
assert r_role.observation_count == 1  # bob -> deployment-operator
assert r_pa.observation_count == 2    # deployment-operator -> deploy, view_logs
assert r_rel.observation_count == 4   # member_of, has_role, has_permission, applies_to
assert r_meta.observation_count == 2  # department via directory + secondary feed
print()

# Every observation evidence row is DERIVED_FROM its run's artifact row.
rel_ok = True
count_obs = 0
for r in pipeline_runs:
    artifact = r.evidence_ids[0]
    for ev_id in r.evidence_ids[1:]:
        rels = app.evidence_store.get_relationships(ev_id)
        ok = any(
            x.relation_type == EvidenceRelation.DERIVED_FROM
            and str(x.target_id) == str(artifact)
            for x in rels
        )
        rel_ok = rel_ok and ok
        count_obs += 1
assert rel_ok and count_obs >= 24, count_obs
print(f"DERIVED_FROM links: {count_obs} observations across {len(pipeline_runs)} artifacts")

# Every row persists as OBSERVED (identity evidence never elevates beyond observed).
stored = {e.id: e.status for e in app.evidence_store.list(limit=10000)}
assert all(stored[ev] == EvidenceStatus.OBSERVED for r in pipeline_runs
           for ev in r.evidence_ids), "identity evidence must stay OBSERVED"
print("All identity evidence rows persisted with status OBSERVED")

# --- world materialization --------------------------------------------------
wm = engine.world_model
entities = wm.list_entities(WorldQuery(mission_id=MID, limit=1000))
etypes = {e.entity_type.value for e in entities}
needed = {"directory", "identity", "group", "role", "permission", "resource"}
assert needed <= etypes, etypes
print("World entity types:", ", ".join(sorted(etypes)))

rels = wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000))
rel_types = {getattr(r.relationship_type, "value", r.relationship_type) for r in rels}
need_rel = {"contains", "member_of", "has_role", "has_permission", "applies_to"}
assert need_rel <= rel_types, rel_types
attack_graph = rel_types & {"exploits", "can_compromise", "leads_to", "enables"}
assert not attack_graph, f"Attack-graph relationships must not be materialized: {attack_graph}"
print("Relationship types:", ", ".join(sorted(rel_types)))
print("No attack-graph relationship types (EXPLOITS/CAN_COMPROMISE/LEADS_TO/ENABLES): PASS")

# Confidence policy: direct inventory observations -> HIGH, derived -> MEDIUM.
_conf = {e.id: e.confidence for e in app.evidence_store.list(limit=10000)}
for ev_id in r_inv.evidence_ids[1:]:
    assert _conf[ev_id] == Confidence.HIGH, ev_id
for ev_id in r_rel.evidence_ids[1:]:
    assert _conf[ev_id] == Confidence.MEDIUM, ev_id
sales = next(o for o in r_meta.observations if o.source != "directory")
assert _conf[r_meta.evidence_ids[r_meta.observations.index(sales) + 1]] == Confidence.MEDIUM
print("Confidence policy (inventory -> HIGH, relationship -> MEDIUM, feed metadata -> MEDIUM): PASS")

---

In [ ]:
# Mission isolation: identity work under a second mission is disjoint.
MID2 = "mission_phase10_id_other"
scope2 = TargetScope(
    mission_id=MID2,
    allowed_targets=[_target(DIR)],
    max_risk_level=RiskLevel.HIGH,
)
req2 = IdentityRequest(
    mission_id=MID2, session_id="ses_phase10_id_2", scope=scope2,
    mode=IdentityMode.CONTROLLED, max_observations=500, timeout_seconds=30.0,
)
res2 = engine.observe_role_assignment(req2, DIR, identity="alice")
other_ids = {str(x) for x in res2.evidence_ids}
assert other_ids.isdisjoint({str(x) for x in r_role.evidence_ids})
assert app.evidence_store.count(MID2) == len(res2.evidence_ids)
assert engine.world_model.count_entities(MID2) >= 1
print("Mission isolation: second mission produced its own evidence/world rows: PASS")

# Redaction: the inventory artifact preserves structure but strips secrets.
_rows = {e.id: e for e in app.evidence_store.list(limit=10000)}
artifact = _rows[r_inv.evidence_ids[0]]
assert artifact.evidence_type == EvidenceType.ARTIFACT
for marker in ("demo-build-secret-hash-0000", "demo-session-token-0000", "demo-api-key-0000"):
    assert marker not in artifact.raw_data, marker
assert "REDACTED" in artifact.raw_data
print("Inventory artifact redacted (no password_hash / session_token / api_key values): PASS")

from blackforge.identity.redaction import (
    credential_value_redacted,
    redact_identity_raw,
)
import json

demo_raw = json.dumps({
    "kind": "identity",
    "identity": "build-service",
    "password_hash": "demo-build-secret-hash-0000",
    "session_token": "demo-session-token-0000",
    "credentials": {"api_key": "demo-api-key-0000"},
    "display_name": "Build Automation Service",
})
clean_raw = redact_identity_raw(demo_raw)
assert "demo-build-secret-hash-0000" not in clean_raw
assert "demo-api-key-0000" not in clean_raw
clean = json.loads(clean_raw)
assert clean["display_name"] == "Build Automation Service"
assert clean["password_hash"] == credential_value_redacted()
assert clean["credentials"] == credential_value_redacted()
print("Redaction unit behavior (credential-like fields -> safe REDACTED marker): PASS")

# Idempotency: a repeated run over the same target reuses the same evidence rows.
before = app.evidence_store.count(MID)
engine.inventory_identities(req, DIR)
after = app.evidence_store.count(MID)
assert after == before, (before, after)
print("Idempotent re-run (no duplicate evidence rows): PASS")

# Metadata contradiction surfaces instead of silently overwriting.
from blackforge.evidence.repository import InMemoryEvidenceRepository
from blackforge.identity.materializer import IdentityWorldMaterializer
from blackforge.identity.normalization import adapter_for_tool
from blackforge.identity.transport import MockIdentityTransport
from blackforge.world_model.repository import InMemoryWorldRepository
from blackforge.world_model.store import WorldModelStore

wm2 = WorldModelStore(repository=InMemoryWorldRepository())
mat = IdentityWorldMaterializer(wm2)
adapter = adapter_for_tool("observe_metadata")
t = MockIdentityTransport()
out = adapter.adapt(
    t.observe_metadata(DIR, mode=IdentityMode.CONTROLLED, identity="alice"),
    context={"target": DIR, "mode": IdentityMode.CONTROLLED},
)
report = mat.materialize(
    MID2,
    [(o, f"ev_meta_{i}", Confidence.HIGH if o.source == "directory" else Confidence.MEDIUM)
     for i, o in enumerate(out.observations)],
    session_id="ses_phase10_id_meta",
)
assert report.assertions_contradicted == 1, report.assertions_contradicted
alice2 = next(e for e in wm2.list_entities(WorldQuery(mission_id=MID2, limit=1000))
              if e.name == "alice")
assertions = wm2.list_assertions(str(alice2.id), lifecycle=None)
amap = {(a.property_key, a.property_value): a.epistemic_status.value for a in assertions}
assert amap[("department", "engineering")] == "observed", amap
assert amap[("department", "sales")] == "inferred", amap
print("Metadata contradiction surfaced (engineering=OBSERVED, sales=INFERRED, contradiction recorded): PASS")

---

In [ ]:
from blackforge.core.errors import AuthorizationError, IdentityExecutionError
from blackforge.identity.models import IdentityStatus

# 1) Target outside the scope is denied BEFORE any transport runs.
#    ("MINECORP" is a supported target type but was never added to the scope.)
denied_out = True
try:
    engine.inventory_identities(req, "MINECORP")
    denied_out = False
except AuthorizationError:
    pass
assert denied_out, "out-of-scope target MINECORP must be denied"
print("Out-of-scope target denied before transport execution: PASS")

# 2) Unsupported target type is rejected (identity supports DIR/ASSET/DOMAIN only).
unsupported_type = True
try:
    engine.inventory_identities(req, "192.0.2.10")
    unsupported_type = False
except IdentityExecutionError:
    pass
assert unsupported_type, "IP target type must be rejected"
print("Unsupported target type rejected (no generic execution surface): PASS")

# 3) Unknown capability is rejected (no generic execution surface).
unknown_rejected = True
try:
    engine.run(req, "identity.not_real", DIR)
    unknown_rejected = False
except IdentityExecutionError:
    pass
assert unknown_rejected, "unknown capability must be rejected"
print("Unknown capability rejected (no generic execution surface): PASS")

# 4) Failure states on the mock error directories.
_error_map = {
    "SNAIL-DIR": IdentityStatus.TIMEOUT,
    "BURSTY-DIR": IdentityStatus.RATE_LIMITED,
    "LOCKED-DIR": IdentityStatus.UNAUTHORIZED,
    "GARBLED-DIR": IdentityStatus.MALFORMED_RESPONSE,
    "FABRICATED-DIR": IdentityStatus.UNSUPPORTED_DIRECTORY,
    "OTHER-CORP": IdentityStatus.UNSUPPORTED_DIRECTORY,
}
for directory, expected in _error_map.items():
    got = engine.inventory_identities(req, directory)
    assert got.status == expected, (directory, got.status, expected)
print("Failure state mapping (6 synthetic error directories) verified: PASS")

# 5) Unknown identity -> structured NO_EVIDENCE, never a crash.
ghost = engine.observe_membership(req, DIR, identity="ghost-identity")
assert ghost.status == IdentityStatus.NO_EVIDENCE
assert "identity not present" in (ghost.error or "")
print("Unknown identity -> NO_EVIDENCE with message: PASS")

# 6) Passive mode is LOW confidence and never collides with controlled records.
from blackforge.core.types import Confidence

req_pas = IdentityRequest(
    mission_id=MID, session_id="ses_phase10_id_pas", scope=scope,
    mode=IdentityMode.PASSIVE, max_observations=500, timeout_seconds=30.0,
)
pas = engine.inventory_identities(req_pas, DIR)
assert pas.mode == IdentityMode.PASSIVE
pas_ids = {str(x) for x in pas.evidence_ids}
assert pas_ids.isdisjoint({str(x) for x in r_inv.evidence_ids})
pas_ev = {e.id: e.confidence for e in app.evidence_store.list(limit=10000)}
assert all(pas_ev[ev] == Confidence.LOW for ev in pas.evidence_ids[1:]), "PASSIVE -> LOW"
print("Confidence mode policy (PASSIVE -> LOW, CONTROLLED direct -> HIGH): PASS")

---

In [ ]:
from blackforge.evidence.repository import SQLiteEvidenceRepository
from blackforge.evidence.store import EvidenceStore
from blackforge.world_model.repository import SQLiteWorldRepository
from blackforge.world_model.store import WorldModelStore

# Fresh connections over the same SQLite files prove restart persistence.
fresh_ev = EvidenceStore(SQLiteEvidenceRepository(str(DBROOT / "evidence.db")))
fresh_wm = WorldModelStore(SQLiteWorldRepository(str(DBROOT / "world_model.db")))

persisted_ev = fresh_ev.count(MID) == app.evidence_store.count(MID)
persisted_wm = fresh_wm.count_entities(MID) == engine.world_model.count_entities(MID)
alice_entity = fresh_wm.find_entity(MID, EntityType.IDENTITY, "alice", namespace="aelionix-corp")
if alice_entity is None:
    alice_entity = next((e for e in fresh_wm.list_entities(WorldQuery(mission_id=MID, limit=1000))
                         if e.entity_type.value == "identity" and e.name == "alice"), None)
persisted_alice = alice_entity is not None
rel_count = len(fresh_wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000)))
assert persisted_ev and persisted_wm and persisted_alice and rel_count > 0
assert fresh_ev.count(MID) > 0
PERSIST_OK = persisted_ev and persisted_wm and persisted_alice

for store in (fresh_ev, fresh_wm):
    store.close()
try:
    app.evidence_store.close()
except Exception:
    pass
try:
    app.world_model.close()
except Exception:
    pass
print("Restart persistence (fresh connections on same DB files): PASS")
print("Backends closed. Validation summary below.")

---

In [ ]:
results = {}
phase_checks = {
    "repository_integrity": (REPO_DIR / "blackforge" / "identity" / "engine.py").exists(),
    "phase10_modules": bool(
        (REPO_DIR / "blackforge" / "identity" / "capabilities.py").exists()
        and (REPO_DIR / "blackforge" / "identity" / "evidence.py").exists()
        and (REPO_DIR / "blackforge" / "identity" / "materializer.py").exists()
        and (REPO_DIR / "blackforge" / "identity" / "redaction.py").exists()
        and (REPO_DIR / "blackforge" / "identity" / "normalization.py").exists()
        and (REPO_DIR / "blackforge" / "identity" / "transport.py").exists()
    ),
    "imports": len(_import_failures) == 0,
    "bootstrap_identity_ready": BOOTSTRAP_OK,
    "capability_surface": CAPS_OK,
    "pipeline_evidence": rel_ok,
    "evidence_observed": bool(count_obs >= 24),
    "world_materialized": bool(needed <= etypes),
    "no_attack_graph": not bool(attack_graph),
    "confidence_policy": True,
    "scope_authorization": denied_out,
    "unsupported_type_rejected": unsupported_type,
    "unknown_capability_rejected": unknown_rejected,
    "redaction_boundary": True,
    "metadata_contradiction": bool(report.assertions_contradicted == 1),
    "mission_isolation": bool(other_ids.isdisjoint({str(x) for x in r_role.evidence_ids})),
    "idempotent_runs": bool(after == before),
    "restart_persistence": PERSIST_OK,
}

# The pytest cell aborts the run on failure, so reaching this cell proves it passed.
pytest_passed = True
install_ok = len(_import_failures) == 0

results["Repository"] = phase_checks["repository_integrity"]
results["Python"] = sys.version_info >= (3, 10)
results["Hardware"] = True  # CPU fallback always works; this notebook needs no GPU
results["Installation"] = install_ok
results["Imports"] = install_ok
results["Automated tests"] = pytest_passed
results["Bootstrap"] = phase_checks["bootstrap_identity_ready"]
results["Phase-specific tests"] = all(phase_checks.values())
results["Security checks"] = (
    phase_checks["scope_authorization"]
    and phase_checks["unsupported_type_rejected"]
    and phase_checks["unknown_capability_rejected"]
    and phase_checks["redaction_boundary"]
    and phase_checks["no_attack_graph"]
    and phase_checks["idempotent_runs"]
)

print()
print("=" * 60)
print("PHASE 10 COLAB VALIDATION SUMMARY")
print("=" * 60)
for name, ok in results.items():
    symbol = "PASS" if ok else "FAIL"
    print(f"  [{symbol}] {name}")

_all_ok = all(results.values()) and all(phase_checks.values())
assert _all_ok, "One or more validation checks failed"

print()
print("LOCAL VALIDATION: SUCCESS")
print()
print("Note: this notebook validates the commit checked out into /content/blackforge.")

---